In [1]:
import os 

In [2]:
import os
import json
import traceback

import pandas as pd
import PyPDF2

from dotenv import load_dotenv

from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
load_dotenv()

True

In [4]:
api = os.getenv("MISTRAL")

In [5]:
print(f"API Key: {api}")

API Key: V7SlGNy2FmomyWAeQVNyhsvHS2czLoC3


In [6]:
llm = ChatMistralAI(
    api_key=api,
    model="mistral-large-latest",
    temperature=1,
)

In [7]:
llm

ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Mistral Large (latest)', 'release_date': '2024-11-01', 'last_updated': '2025-12-02', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 262144, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': True, 'temperature': True}, client=<httpx.Client object at 0x0000025C1F7F34D0>, async_client=<httpx.AsyncClient object at 0x0000025C1F800050>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='mistral-large-latest', temperature=1.0, model_kwargs={})

In [8]:
with open("../response.json", "r") as f:
    response = json.load(f)

In [9]:
print(response)

{'quiz': [{'MCQS': 'Question 1', 'options': {'A': 'Option A', 'B': 'Option B', 'C': 'Option C', 'D': 'Option D'}, 'correct_answer': 'B'}, {'MCQS ': 'Question 2', 'options': {'A': 'Option A', 'B': 'Option B', 'C': 'Option C', 'D': 'Option D'}, 'correct_answer': 'C'}]}


In [10]:
template = """
You are an expert MCQ creator with strong knowledge of educational assessment.

Your task is to create a quiz based ONLY on the provided text.

TEXT:
{text}

Instructions:
- Generate exactly {number} multiple-choice questions.
- multiple choice question from {subject} subject.
- The difficulty level should be {tone}.
- Every question must have exactly four options labeled A, B, C, and D..
- Only one option should be correct.
- Do not create questions from information that is not present in the provided text.
- Ensure the questions are clear, relevant, and non-repetitive.
- Make sure the quiz is suitable for students.
- Return ONLY valid JSON.
- Do not include explanations, markdown, or additional text.

The JSON format must exactly match the following schema:

{response}
"""

In [11]:
prompt = PromptTemplate (
    input_variables=["text", "number", "subject", "tone", "response"],
    template=template,
)

In [12]:
chain1 = prompt | llm 

In [13]:
print(chain1)

first=PromptTemplate(input_variables=['number', 'response', 'subject', 'text', 'tone'], input_types={}, partial_variables={}, template='\nYou are an expert MCQ creator with strong knowledge of educational assessment.\n\nYour task is to create a quiz based ONLY on the provided text.\n\nTEXT:\n{text}\n\nInstructions:\n- Generate exactly {number} multiple-choice questions.\n- multiple choice question from {subject} subject.\n- The difficulty level should be {tone}.\n- Every question must have exactly four options labeled A, B, C, and D..\n- Only one option should be correct.\n- Do not create questions from information that is not present in the provided text.\n- Ensure the questions are clear, relevant, and non-repetitive.\n- Make sure the quiz is suitable for students.\n- Return ONLY valid JSON.\n- Do not include explanations, markdown, or additional text.\n\nThe JSON format must exactly match the following schema:\n\n{response}\n') middle=[] last=ChatMistralAI(metadata={'lc_versions': {

In [24]:
template2 = """You are an expert English examiner and MCQ evaluator.

Subject:
{subject}


Task:
You must evaluate the given multiple choice questions based on cognitive and analytical level of students.

Instructions:
- Check if questions are clear, correct, and meaningful
- Check difficulty level according to students' understanding
- Identify any grammar or conceptual mistakes
- If quiz is not appropriate for students' cognitive level, suggest regeneration

Final Output Rules:
1. Provide a complete analysis in exactly 50 words
2. Clearly state whether quiz is GOOD or NEEDS IMPROVEMENT
3. If needed, suggest to regenerate improved version
4. Be strict and professional like an examiner
Quiz MCQs:
{quiz}

"""

In [25]:
prompt2 = PromptTemplate (
    input_variables=["subject","quiz"],
    template=template2,
)

In [26]:
chain2 = prompt2 | llm 

In [16]:
with open("../data.txt", "r") as f:
    text = f.read()

In [ ]:
text 
numbers = 10 
subject = "AI"
tone = "medium"
response = response 
